# skforecast-ai — Demo Phase 4

This notebook demonstrates **Phase 4: Code Generation**.

Given a `ForecastPlan` and a `DataProfile`, the `generate_code` function
produces a complete, executable Python script ready to run with the user's data.
No LLM is involved — templates are pure deterministic string construction.

## Setup

In [1]:
import numpy as np
import pandas as pd

import skforecast_ai
from skforecast_ai.profiling import create_data_profile
from skforecast_ai.recommendation import recommend_plan
from skforecast_ai.generation import generate_code

print(f"skforecast-ai version: {skforecast_ai.__version__}")

skforecast-ai version: 0.1.0


## 1. Single series — ForecasterRecursive (no exog)

In [2]:
df_daily = pd.DataFrame(
    {"y": np.random.default_rng(42).normal(100, 10, 365)},
    index=pd.date_range("2023-01-01", periods=365, freq="D"),
)

profile = create_data_profile(data=df_daily, target="y")
plan = recommend_plan(profile=profile, horizon=30)

print(f"Task: {plan.task_type} | Forecaster: {plan.forecaster} | Estimator: {plan.estimator}")
print(f"Lags: {plan.lags} | Interval: {plan.interval_method}")

Task: single_series | Forecaster: ForecasterRecursive | Estimator: Ridge
Lags: [1, 2, 3, 4, 5, 6, 7] | Interval: bootstrapping


In [3]:
code = generate_code(plan=plan, profile=profile, data_path="daily_sales.csv")
print(code)

import pandas as pd
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from sklearn.linear_model import Ridge

# Load data
data = pd.read_csv('daily_sales.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Train/test split (time-based)
n_train = int(len(data) * 0.8)
y_train = data.iloc[:n_train]['y']
y_test = data.iloc[n_train:]['y']

# Create forecaster
forecaster = ForecasterRecursive(
    estimator = Ridge(random_state=123),
    lags      = [1, 2, 3, 4, 5, 6, 7],
)

# Fit
forecaster.fit(y=y_train)

# Predict
predictions = forecaster.predict(steps=30)
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps              = 30,
    initial_train_size = n_train,
    refit              = False,
)

metric, predictions_bt = backtesting_forecaster(
    forecaster = forecaster,
    y          = data['y'],
    cv         = cv,
    metric     = 'mean_absolute_error',
)
print(f"Backtest {metric}

## 2. Single series with exogenous variables

In [4]:
rng = np.random.default_rng(123)
df_exog = pd.DataFrame(
    {
        "sales": rng.normal(500, 50, 720),
        "temperature": rng.normal(20, 5, 720),
        "promo_budget": rng.uniform(1000, 5000, 720),
    },
    index=pd.date_range("2023-01-01", periods=720, freq="h"),
)

profile_exog = create_data_profile(data=df_exog, target="sales")
plan_exog = recommend_plan(profile=profile_exog, horizon=24)

print(f"Use exog: {plan_exog.use_exog}")
print(f"Exog columns: {profile_exog.exog_columns}")

Use exog: True
Exog columns: ['temperature', 'promo_budget']


In [5]:
code_exog = generate_code(plan=plan_exog, profile=profile_exog, data_path="hourly_data.csv")
print(code_exog)

import pandas as pd
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from lightgbm import LGBMRegressor

# Load data
data = pd.read_csv('hourly_data.csv', index_col=0, parse_dates=True)
data = data.asfreq('h')

# Train/test split (time-based)
n_train = int(len(data) * 0.8)
y_train = data.iloc[:n_train]['sales']
y_test = data.iloc[n_train:]['sales']
exog_train = data.iloc[:n_train][['temperature', 'promo_budget']]
exog_test = data.iloc[n_train:][['temperature', 'promo_budget']]

# Create forecaster
forecaster = ForecasterRecursive(
    estimator = LGBMRegressor(random_state=123),
    lags      = [1, 2, 3, 4, 5, 6, 7, 24, 168],
)

# Fit
forecaster.fit(y=y_train, exog=exog_train)

# Predict
predictions = forecaster.predict(steps=24, exog=exog_test.iloc[:24])
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps              = 24,
    initial_train_size = n_train,
    refit              = False,
)


## 3. Multi-series (long format)

In [6]:
rng = np.random.default_rng(99)
dates = pd.date_range("2023-01-01", periods=300, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates, 3),
    "series_id": np.repeat(["store_A", "store_B", "store_C"], 300),
    "sales": rng.normal(200, 30, 900),
})

profile_multi = create_data_profile(
    data=df_multi, target="sales", date_column="date", series_id_column="series_id"
)
plan_multi = recommend_plan(profile=profile_multi, horizon=14)

print(f"Task: {plan_multi.task_type} | Forecaster: {plan_multi.forecaster}")

Task: multi_series | Forecaster: ForecasterRecursiveMultiSeries


In [7]:
code_multi = generate_code(plan=plan_multi, profile=profile_multi, data_path="stores.csv")
print(code_multi)

import pandas as pd
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.model_selection import backtesting_forecaster_multiseries, TimeSeriesFold
from lightgbm import LGBMRegressor

# Load data (long format)
data = pd.read_csv('stores.csv', parse_dates=['date'])

# Pivot to wide format (columns = series)
series = data.pivot_table(
    index='date', columns='series_id', values='sales'
)
series.index = pd.DatetimeIndex(series.index)
series.index.name = None

# Train/test split
n_train = int(len(series) * 0.8)

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator = LGBMRegressor(random_state=123),
    lags      = [1, 2, 3, 4, 5, 6, 7],
    encoding  = 'ordinal',
)

# Fit
forecaster.fit(series=series)

# Predict
predictions = forecaster.predict(steps=14)
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = n_train,
    refit              = False,
)

metric, predictions_bt = backtes

## 4. Statistical model (user preference)

In [8]:
plan_stats = recommend_plan(profile=profile, horizon=30, prefer_statistical=True)

print(f"Task: {plan_stats.task_type} | Forecaster: {plan_stats.forecaster}")
print(f"Estimator: {plan_stats.estimator} | Lags: {plan_stats.lags}")

Task: statistical | Forecaster: ForecasterStats
Estimator: None | Lags: None


In [9]:
code_stats = generate_code(plan=plan_stats, profile=profile, data_path="monthly_revenue.csv")
print(code_stats)

import pandas as pd
from skforecast.stats import ForecasterStats, Arima
from skforecast.model_selection import backtesting_stats, TimeSeriesFold

# Load data
data = pd.read_csv('monthly_revenue.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Train/test split (time-based)
n_train = int(len(data) * 0.8)
y_train = data.iloc[:n_train]['y']
y_test = data.iloc[n_train:]['y']

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal=True),
)

# Fit
forecaster.fit(y=y_train)

# Predict
predictions = forecaster.predict(steps=30)
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps              = 30,
    initial_train_size = n_train,
    refit              = False,
)

metric, predictions_bt = backtesting_stats(
    forecaster = forecaster,
    y          = data['y'],
    cv         = cv,
    metric     = 'mean_absolute_error',
)
print(f"Backtest {metric}")

# Prediction intervals (native)
predictions_interval = forecast

## 5. Foundation model (user preference)

In [10]:
plan_found = recommend_plan(profile=profile, horizon=30, prefer_foundation=True)

print(f"Task: {plan_found.task_type} | Forecaster: {plan_found.forecaster}")
print(f"Estimator: {plan_found.estimator} | Lags: {plan_found.lags}")

Task: foundation | Forecaster: ForecasterFoundation
Estimator: None | Lags: None


In [11]:
code_found = generate_code(plan=plan_found, profile=profile, data_path="energy.csv")
print(code_found)

import pandas as pd
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.model_selection import backtesting_foundation, TimeSeriesFold

# Load data
data = pd.read_csv('energy.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Create foundation model (Chronos-2)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 2048,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=data['y'])

# Predict
predictions = forecaster.predict(steps=30)
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps              = 30,
    initial_train_size = int(len(data) * 0.8),
    refit              = False,
)

metric, predictions_bt = backtesting_foundation(
    forecaster = forecaster,
    series     = data['y'],
    cv         = cv,
    metric     = 'mean_absolute_error',
)
print(f"Backtest {me

## 6. Syntax validation

All generated scripts must be syntactically valid Python. Let's verify:

In [12]:
all_codes = {
    "single_series (no exog)": code,
    "single_series (with exog)": code_exog,
    "multi_series": code_multi,
    "statistical": code_stats,
    "foundation": code_found,
}

for name, script in all_codes.items():
    try:
        compile(script, f"<{name}>", "exec")
        print(f"  {name}: OK")
    except SyntaxError as e:
        print(f"  {name}: FAILED - {e}")

  single_series (no exog): OK
  single_series (with exog): OK
  multi_series: OK
  statistical: OK
  foundation: OK


## 7. Error handling — unsupported task type

In [13]:
from skforecast_ai.schemas import ForecastPlan, DataProfile

plan_unsupported = ForecastPlan(
    task_type="multivariate",
    forecaster="ForecasterDirectMultiVariate",
    estimator="Ridge",
    horizon=10,
    frequency="D",
    lags=[1, 2, 3],
    metric="mean_absolute_error",
    backtesting_strategy="TimeSeriesFold",
    interval_method=None,
    use_exog=False,
    rationale="Test unsupported.",
)

try:
    generate_code(plan=plan_unsupported, profile=profile)
except ValueError as e:
    print(f"ValueError caught: {e}")

ValueError caught: Unsupported task_type 'multivariate'. Supported types: ['single_series', 'multi_series', 'statistical', 'foundation']


## Summary

Phase 4 provides `generate_code(plan, profile, data_path)` which:

- Dispatches to the correct template based on `plan.task_type`
- Handles 4 template types: single series, multi-series, statistical, foundation
- Includes conditional exog handling and prediction intervals
- Raises `ValueError` for unsupported task types
- Always produces syntactically valid Python